In [1]:
import boto3
import json

# Crear clientes
ec2 = boto3.client('ec2')
s3 = boto3.client('s3')
lambda_client = boto3.client('lambda')


In [2]:

def obtener_ec2():
    instancias = []
    try:
        response = ec2.describe_instances()

        for reserva in response['Reservations']:
            for instancia in reserva['Instances']:
                instancias.append({
                    "InstanceId": instancia.get("InstanceId"),
                    "Tipo": instancia.get("InstanceType"),
                    "Estado": instancia.get("State", {}).get("Name"),
                    "IP_Publica": instancia.get("PublicIpAddress")
                })
    except Exception as e:
        print("Error EC2:", e)

    return instancias


def obtener_s3():
    buckets = []
    try:
        response = s3.list_buckets()

        for bucket in response['Buckets']:
            buckets.append({
                "Nombre": bucket.get("Name"),
                "FechaCreacion": str(bucket.get("CreationDate"))
            })
    except Exception as e:
        print("Error S3:", e)

    return buckets


def obtener_lambda():
    funciones = []
    try:
        response = lambda_client.list_functions()

        for func in response['Functions']:
            funciones.append({
                "Nombre": func.get("FunctionName"),
                "Runtime": func.get("Runtime"),
                "Estado": func.get("State")
            })
    except Exception as e:
        print("Error Lambda:", e)

    return funciones


def generar_inventario():
    inventario = {
        "EC2": obtener_ec2(),
        "S3": obtener_s3(),
        "Lambda": obtener_lambda()
    }

    # Guardar en archivo JSON
    with open("aws_inventory.json", "w") as f:
        json.dump(inventario, f, indent=4)

    print("Inventario generado en aws_inventory.json")

In [ ]:

if __name__ == "__main__":
    generar_inventario()